In [22]:
!pip install -q gradio google-generativeai

In [23]:
!pip install -q gradio langchain-google-genai langchain-core python-dotenv

In [24]:
# Run this cell, click "Choose Files", upload crop_yield.csv
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

Saving archive (1).csv to archive (1) (1).csv
Uploaded: ['archive (1) (1).csv']


In [25]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/archive (1).csv')

# Standardise column names
df.columns = df.columns.str.strip().str.title().str.replace(' ', '_')

rename_map = {}
for col in df.columns:
    cl = col.lower()
    if 'state' in cl:            rename_map[col] = 'State'
    elif 'crop' in cl and 'year' not in cl: rename_map[col] = 'Crop'
    elif 'year' in cl:           rename_map[col] = 'Year'
    elif 'season' in cl:         rename_map[col] = 'Season'
    elif 'area' in cl:           rename_map[col] = 'Area'
    elif 'production' in cl:     rename_map[col] = 'Production'
    elif 'yield' in cl:          rename_map[col] = 'Yield'
df.rename(columns=rename_map, inplace=True)

for col in ['State', 'Crop', 'Season']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.title()

for col in ['Area', 'Production', 'Yield']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df['Year'] = pd.to_numeric(df['Year'], errors='coerce').astype('Int64')
df.dropna(subset=['Production', 'Area'], how='all', inplace=True)
df.reset_index(drop=True, inplace=True)

print('✅ Loaded. Shape:', df.shape)
print('Columns:', df.columns.tolist())
print('Years:', df['Year'].min(), '–', df['Year'].max())
print(df.head(3))

✅ Loaded. Shape: (19689, 10)
Columns: ['Crop', 'Year', 'Season', 'State', 'Area', 'Production', 'Annual_Rainfall', 'Fertilizer', 'Pesticide', 'Yield']
Years: 1997 – 2020
          Crop  Year      Season  State     Area  Production  Annual_Rainfall  \
0     Arecanut  1997  Whole Year  Assam  73814.0       56708           2051.4   
1    Arhar/Tur  1997      Kharif  Assam   6637.0        4685           2051.4   
2  Castor Seed  1997      Kharif  Assam    796.0          22           2051.4   

   Fertilizer  Pesticide     Yield  
0  7024878.38   22882.34  0.796087  
1   631643.29    2057.47  0.710435  
2    75755.32     246.76  0.238333  


In [38]:
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GOOGLE_API_KEY')

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GEMINI_API_KEY,
    temperature=0
)

print('✅ Gemini ready')

✅ Gemini ready


In [39]:
from langchain_core.prompts import ChatPromptTemplate

# ── Define schema exactly like your Samsung review example ────────────────────
query_schema = {
    "title": "QueryPlan",
    "type": "object",
    "properties": {
        "answerable": {
            "type": "boolean",
            "description": "Can this question be answered from the Indian crop production dataset (1997-2020)? False if question is about crime, health, weather, GDP, or any year outside 1997-2020"
        },
        "reason_if_not": {
            "type": ["string", "null"],
            "description": "If answerable is false, explain why. Otherwise null."
        },
        "operation": {
            "type": "string",
            "enum": ["top_n", "trend", "compare", "aggregate", "filter_show"],
            "description": "top_n for best/most/highest, trend for over time questions, compare for side by side, aggregate for total/average"
        },
        "filters": {
            "type": "object",
            "description": "What to filter the data by",
            "properties": {
                "State":  {"type": ["string", "null"], "description": "Indian state name or null"},
                "Crop":   {"type": ["string", "null"], "description": "Crop name like Wheat, Rice, Sugarcane or null"},
                "Year":   {"type": ["integer", "null"], "description": "A year between 1997 and 2020 or null"},
                "Season": {"type": ["string", "null"], "description": "Kharif, Rabi, Whole Year, Summer, Winter, Autumn or null"}
            }
        },
        "metric": {
            "type": "string",
            "enum": ["Production", "Area", "Yield"],
            "description": "Production in tonnes, Area in hectares, Yield in tonnes/hectare"
        },
        "group_by": {
            "type": ["string", "null"],
            "enum": ["State", "Crop", "Year", "Season", None],
            "description": "What column to group results by. Use Year for trends, State for state comparisons"
        },
        "sort_order": {
            "type": "string",
            "enum": ["desc", "asc"],
            "description": "desc for highest first, asc for lowest first"
        },
        "n": {
            "type": ["integer", "null"],
            "description": "How many top results to return. Default 5."
        },
        "aggregation": {
            "type": "string",
            "enum": ["sum", "mean", "max", "min"],
            "description": "How to aggregate numbers. sum for total, mean for average"
        }
    },
    "required": ["answerable", "operation", "filters", "metric", "sort_order", "aggregation"]
}

# ── Attach schema to model — exactly like your example ────────────────────────
structured_llm = llm.with_structured_output(query_schema)

# ── Prompt ────────────────────────────────────────────────────────────────────
PROMPT = ChatPromptTemplate.from_messages([
    ("system", """
You are a query planner for an Indian crop production dataset (1997–2020).

Dataset columns:
- State      : Indian state (e.g. Punjab, West Bengal, Maharashtra)
- Crop       : Crop name (e.g. Wheat, Rice, Sugarcane, Cotton, Maize)
- Year       : 1997 to 2020 only
- Season     : Kharif, Rabi, Whole Year, Summer, Winter, Autumn
- Area       : Area cultivated in hectares
- Production : Total production in tonnes
- Yield      : Yield in tonnes per hectare

Rules:
- Questions about crime, weather, health, GDP → answerable=false
- Year outside 1997-2020 → answerable=false
- "trend/over time/from X to Y" → operation=trend, group_by=Year
- "which state most/highest" → operation=top_n, group_by=State
- "compare crops" → operation=compare, group_by=Crop
- "total/average" → operation=aggregate, group_by=null
"""),
    ("human", "{question}")
])

# ── Chain exactly like your Samsung example ───────────────────────────────────
chain = PROMPT | structured_llm

def get_query_plan(question: str) -> dict:
    result = chain.invoke({"question": question})
    return result   # already a dict — no .model_dump() needed since we used JSON schema

# ── Test it ───────────────────────────────────────────────────────────────────
import json
plan = get_query_plan("Which state produced the most wheat in 2015?")
print(json.dumps(plan, indent=2))

{
  "answerable": true,
  "operation": "top_n",
  "filters": {
    "State": null,
    "Crop": "Wheat",
    "Year": 2015,
    "Season": null
  },
  "metric": "Production",
  "group_by": "State",
  "sort_order": "desc",
  "n": 1,
  "aggregation": "sum"
}


In [40]:
def execute_plan(plan, df):
    steps = []
    data = df.copy()
    filters = plan.get('filters', {})

    if filters.get('State'):
        val = filters['State'].title()
        matches = [s for s in data['State'].unique() if val.lower() in s.lower()]
        if not matches:
            return None, f"State '{val}' not found in dataset."
        data = data[data['State'].isin(matches)]
        steps.append(f"Filter State = {matches}")

    if filters.get('Crop'):
        val = filters['Crop'].title()
        matches = [c for c in data['Crop'].unique() if val.lower() in c.lower()]
        if not matches:
            return None, f"Crop '{val}' not found in dataset."
        data = data[data['Crop'].isin(matches)]
        steps.append(f"Filter Crop = {matches}")

    if filters.get('Year'):
        data = data[data['Year'] == int(filters['Year'])]
        steps.append(f"Filter Year = {filters['Year']}")

    if filters.get('Season'):
        val = filters['Season'].title()
        data = data[data['Season'].str.lower() == val.lower()]
        steps.append(f"Filter Season = {val}")

    if data.empty:
        return None, "No data found for this combination."

    metric     = plan.get('metric', 'Production')
    group_by   = plan.get('group_by')
    agg_func   = plan.get('aggregation', 'sum')
    sort_order = plan.get('sort_order', 'desc')
    n          = plan.get('n') or 5

    if group_by:
        result = data.groupby(group_by)[metric].agg(agg_func).reset_index()
        result.columns = [group_by, metric]
        steps.append(f"Group by {group_by}, {agg_func} of {metric}")
    else:
        total = data[metric].agg(agg_func)
        result = pd.DataFrame({metric: [total]})
        steps.append(f"{agg_func} of {metric}")

    ascending = (sort_order == 'asc')
    result = result.sort_values(metric, ascending=ascending)

    if plan.get('operation') == 'top_n' and group_by:
        result = result.head(n)
        steps.append(f"Top {n}")
    print('✅ Executor ready')
    return result, ' → '.join(steps)

print('✅ Executor ready')

✅ Executor ready


In [41]:
def build_answer(result_df, plan, question, steps):
    metric   = plan.get('metric', 'Production')
    group_by = plan.get('group_by')
    operation= plan.get('operation')
    units    = {'Production': 'tonnes', 'Area': 'hectares', 'Yield': 'tonnes/hectare'}
    unit     = units.get(metric, '')

    if result_df is None:
        return f"⚠️ {steps}"

    lines = []
    if group_by is None:
        val = result_df[metric].iloc[0]
        lines.append(f"**Result:** {val:,.2f} {unit}")
    elif operation == 'trend':
        first, last = result_df.iloc[0], result_df.iloc[-1]
        change = last[metric] - first[metric]
        direction = "increased" if change > 0 else "decreased"
        lines.append(f"**Trend {int(first[group_by])}–{int(last[group_by])}:**")
        lines.append(f"  • Start: {first[metric]:,.0f} {unit}")
        lines.append(f"  • End:   {last[metric]:,.0f} {unit}")
        lines.append(f"  • Overall {direction} by {abs(change):,.0f} {unit}")
    else:
        lines.append(f"**Top results ({metric}):**")
        for rank, (_, row) in enumerate(result_df.iterrows(), 1):
            lines.append(f"  {rank}. {row[group_by]}: {row[metric]:,.0f} {unit}")

    lines.append(f"\n**How computed:** {steps}")
    return '\n'.join(lines)

print('✅ Answer builder ready')

✅ Answer builder ready


In [44]:
def answer_question(question):
    question = question.strip()
    if not question:
        return "Please enter a question."          # ← removed , None
    try:
        plan = get_query_plan(question)
    except Exception as e:
        return f"❌ Gemini error: {e}"             # ← removed , None

    if not plan.get('answerable', True):
        reason = plan.get('reason_if_not', 'Not in dataset.')
        return f"❌ Out of scope: {reason}\n\n(Dataset covers Indian crop production, 1997–2020)"  # ← removed , None

    try:
        result_df, steps = execute_plan(plan, df)
    except Exception as e:
        return f"❌ Query error: {e}"             # ← removed , None

    if result_df is None:
        return f"⚠️ {steps}"                      # ← removed , None

    answer = build_answer(result_df, plan, question, steps)
    return answer

# # Test
ans = answer_question('Which crop has the highest average yield in Maharashtra?')
print(ans)

✅ Executor ready
**Top results (Yield):**
  1. Sugarcane: 68 tonnes/hectare

**How computed:** Filter State = ['Maharashtra'] → Group by Crop, mean of Yield → Top 1


In [45]:
# ── DELIVERABLE 4: Evaluation Set + Results ───────────────────────────────────

eval_set = [
    {
        "q": "What is the crime rate in Delhi?",
        "expected": "REFUSED — out of scope (crime not in this dataset)",
        "type": "🚫 out-of-scope trap"
    },
    {
        "q": "What was rice production in 2025?",
        "expected": "REFUSED — year 2025 is outside dataset range 1997–2020",
        "type": "🚫 out-of-scope year trap"
    },
    {
        "q": "Which state produced the most wheat in 2015?",
        "expected": "Uttar Pradesh — HAND VERIFIED (checked raw CSV: UP wheat 2015 = ~26M tonnes)",
        "type": "top_n — hand verified ✅"
    },
    {
        "q": "Show the trend of rice production in Punjab from 1997 to 2020",
        "expected": "Year-wise rice production values for Punjab showing upward trend",
        "type": "trend"
    },
]

print('=' * 70)
print('EVALUATION SET + RESULTS')
print('=' * 70)

passed = 0
failed = 0

for i, item in enumerate(eval_set, 1):
    ans= answer_question(item['q'])

    # Auto-check: did out-of-scope questions get refused?
    if '🚫' in item['type']:
        status = '✅ CORRECTLY REFUSED' if '❌ Out of scope' in ans or 'REFUSED' in ans.upper() else '❌ SHOULD HAVE BEEN REFUSED'
    else:
        status = '✅ ANSWERED' if '❌' not in ans[:5] else '❌ FAILED'

    if '✅' in status:
        passed += 1
    else:
        failed += 1

    print(f"\nQ{i} [{item['type']}]")
    print(f"Question : {item['q']}")
    print(f"Expected : {item['expected']}")
    print(f"Got      : {ans[:200]}")
    print(f"Status   : {status}")
    print('-' * 70)

print(f"\n📊 SUMMARY: {passed}/{len(eval_set)} passed, {failed} failed")

EVALUATION SET + RESULTS

Q1 [🚫 out-of-scope trap]
Question : What is the crime rate in Delhi?
Expected : REFUSED — out of scope (crime not in this dataset)
Got      : ❌ Out of scope: The question is about crime rate, which is not available in the crop production dataset.

(Dataset covers Indian crop production, 1997–2020)
Status   : ✅ CORRECTLY REFUSED
----------------------------------------------------------------------

Q2 [🚫 out-of-scope year trap]
Question : What was rice production in 2025?
Expected : REFUSED — year 2025 is outside dataset range 1997–2020
Got      : ❌ Out of scope: Year 2025 is outside the dataset's range (1997-2020).

(Dataset covers Indian crop production, 1997–2020)
Status   : ✅ CORRECTLY REFUSED
----------------------------------------------------------------------
✅ Executor ready

Q3 [top_n — hand verified ✅]
Question : Which state produced the most wheat in 2015?
Expected : Uttar Pradesh — HAND VERIFIED (checked raw CSV: UP wheat 2015 = ~26M tonnes)
Got  

In [47]:
eval=[{
        "q": "Which are the top 5 states by total sugarcane production?",
        "expected": "Uttar Pradesh likely #1, Maharashtra #2",
        "type": "top_n"
    },
    {
        "q": "What was the total area under cotton cultivation in 2010?",
        "expected": "A number in hectares (all states combined)",
        "type": "aggregate"
    },
    {
        "q": "Which crop has the highest average yield in Maharashtra?",
        "expected": "A crop name with yield in tonnes/hectare",
        "type": "top_n by yield"
    },
    {
        "q": "Compare maize production across all states",
        "expected": "Ranked list of states by maize production",
        "type": "compare"
    },
]
print('=' * 70)
print('EVALUATION SET + RESULTS')
print('=' * 70)

passed = 0
failed = 0

for i, item in enumerate(eval_set, 1):
    ans= answer_question(item['q'])

    # Auto-check: did out-of-scope questions get refused?
    if '🚫' in item['type']:
        status = '✅ CORRECTLY REFUSED' if '❌ Out of scope' in ans or 'REFUSED' in ans.upper() else '❌ SHOULD HAVE BEEN REFUSED'
    else:
        status = '✅ ANSWERED' if '❌' not in ans[:5] else '❌ FAILED'

    if '✅' in status:
        passed += 1
    else:
        failed += 1

    print(f"\nQ{i} [{item['type']}]")
    print(f"Question : {item['q']}")
    print(f"Expected : {item['expected']}")
    print(f"Got      : {ans[:200]}")
    print(f"Status   : {status}")
    print('-' * 70)

print(f"\n📊 SUMMARY: {passed}/{len(eval_set)} passed, {failed} failed")

EVALUATION SET + RESULTS

Q1 [🚫 out-of-scope trap]
Question : What is the crime rate in Delhi?
Expected : REFUSED — out of scope (crime not in this dataset)
Got      : ❌ Out of scope: The question is about crime rate, which is not available in the crop production dataset.

(Dataset covers Indian crop production, 1997–2020)
Status   : ✅ CORRECTLY REFUSED
----------------------------------------------------------------------

Q2 [🚫 out-of-scope year trap]
Question : What was rice production in 2025?
Expected : REFUSED — year 2025 is outside dataset range 1997–2020
Got      : ❌ Out of scope: Year 2025 is outside the dataset's range (1997-2020).

(Dataset covers Indian crop production, 1997–2020)
Status   : ✅ CORRECTLY REFUSED
----------------------------------------------------------------------
✅ Executor ready

Q3 [top_n — hand verified ✅]
Question : Which state produced the most wheat in 2015?
Expected : Uttar Pradesh — HAND VERIFIED (checked raw CSV: UP wheat 2015 = ~26M tonnes)
Got  